# Remote Forest Biomass Mapping by random forest models in GEE

In [1]:
# import the libraries
import ee
import pandas as pd
import os
import numpy as np
import random
from random import sample
import itertools 
import geopandas as gpd
from sklearn.metrics import r2_score
from termcolor import colored # this is allocate colour and fonts type for the print title and text
from IPython.display import display, HTML

In [2]:
#set the working directory of local drive for Grid search result table loading
# os.getcwd()
os.chdir('/Users/LeonidMoore/Desktop/BIOMASS') #os.chdir('~/Github_Folder') # check the path 

In [3]:
# initialize the earth engine API
ee.Initialize()

## STEP 1 Data preperation and objects definition

In [5]:
# check the parameter from grid search
# load the grid searh table from R
parameterTable = pd.read_csv('GridSearchResults_SD1/Remote_Sensing_Biomass_Modeling_Grid_Search_Seed_0.csv', float_precision='round_trip')
# show the structure by head function
print(colored('The head of the table: \n', 'blue', attrs=['bold']))
parameterTable.head()

The head of the table: 



,Unnamed: 0,Mean_R2,StDev_R2,ModelName,numberOfTrees,variablesPerSplit,minLeafPopulation,bagFraction,maxNodes,type
0,0,0.874159,0.017559,GridSeach_Model_18_8_90,200,18,8,0.632,90,Classifier.smileRandomForest


In [6]:
# load the biomass map 
biomassDensityMap = ee.Image("users/leonidmoore/ForestBiomass/RemoteSensingModel/20201_ESA_AGB_Carbon_Density_Map").rename('RemoteBiomass')
# load the hansen forest cover, and transform to 0-1
hansenTreeCover = ee.Image("projects/crowtherlab/Composite/CrowtherLab_Composite_30ArcSec").select('HansenEtAl_TreeCover_Year2010').divide(100).rename('PresentTreeCover')

In [8]:
# read the composite
compositeImageRaw = ee.Image("users/leonidmoore/ForestBiomass/20200915_Forest_Biomass_Predictors_Image").addBands(biomassDensityMap).addBands(hansenTreeCover)
# show the band names of the composite image 
print('Composite Band Names:',compositeImageRaw.bandNames().getInfo())

Composite Band Names: ['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvCloudCover_MODCF_interannualSD', 'EarthEnvCloudCover_MODCF_intraannualSD', 'EarthEnvCloudCover_MODCF_meanannual', 'EarthEnvTopoMed_AspectCosine', 'EarthEnvTopoMed_AspectSine', 'EarthE

In [9]:
# define the boundary geography reference
unboundedGeo = ee.Geometry.Polygon([-180, 88, 0, 88, 180, 88, 180, -88, 0, -88, -180, -88], None, False)

In [10]:
# define the list of predictors
propertyOfInterest = ['Aridity_Index',
                      'CHELSA_Annual_Mean_Temperature',
                      'CHELSA_Annual_Precipitation',
                      'CHELSA_Isothermality',
                      'CHELSA_Max_Temperature_of_Warmest_Month',
                      'CHELSA_Mean_Diurnal_Range',
                      'CHELSA_Mean_Temperature_of_Coldest_Quarter',
                      'CHELSA_Mean_Temperature_of_Driest_Quarter',
                      'CHELSA_Mean_Temperature_of_Warmest_Quarter',
                      'CHELSA_Mean_Temperature_of_Wettest_Quarter',
                      'CHELSA_Min_Temperature_of_Coldest_Month',
                      'CHELSA_Precipitation_Seasonality',
                      'CHELSA_Precipitation_of_Coldest_Quarter',
                      'CHELSA_Precipitation_of_Driest_Month',
                      'CHELSA_Precipitation_of_Driest_Quarter',
                      'CHELSA_Precipitation_of_Warmest_Quarter',
                      'CHELSA_Precipitation_of_Wettest_Month',
                      'CHELSA_Precipitation_of_Wettest_Quarter',
                      'CHELSA_Temperature_Annual_Range',
                      'CHELSA_Temperature_Seasonality',
                      'Depth_to_Water_Table',
                      'EarthEnvTopoMed_Eastness',
                      'EarthEnvTopoMed_Elevation',
                      'EarthEnvTopoMed_Northness',
                      'EarthEnvTopoMed_ProfileCurvature',
                      'EarthEnvTopoMed_Roughness',
                      'EarthEnvTopoMed_Slope',
                      'SG_Absolute_depth_to_bedrock',
                      'WorldClim2_SolarRadiation_AnnualMean',
                      'WorldClim2_WindSpeed_AnnualMean',
                      'EarthEnvCloudCover_MODCF_interannualSD',
                      'EarthEnvCloudCover_MODCF_intraannualSD',
                      'EarthEnvCloudCover_MODCF_meanannual',
                      'EarthEnvTopoMed_AspectCosine',
                      'EarthEnvTopoMed_AspectSine',
                      'LandCoverClass_Cultivated_and_Managed_Vegetation',
                      'Human_Disturbance',
                      'LandCoverClass_Urban_Builtup',
                      'SG_Clay_Content_0_100cm',
                      'SG_Coarse_fragments_0_100cm',
                      'SG_Sand_Content_0_100cm',
                      'SG_Silt_Content_0_100cm',
                      'SG_Soil_pH_H2O_0_100cm',
                      'WDPA',
                      'cropland',
                      'grazing',
                      'pasture',
                      'rangeland',
                      'PresentTreeCover']
print(propertyOfInterest[0:5])

['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month']


### STEP 2 Mapping Potential Biomass on Google earth engine

#### 2.1 prepare the composite for potential Mapping

In [11]:
# toggle these human activity layers into zero activity situation
toggledCultivated = compositeImageRaw.select('LandCoverClass_Cultivated_and_Managed_Vegetation').lt(0)
toggledUrban = compositeImageRaw.select('LandCoverClass_Urban_Builtup').lt(0)
toggledDisturbance = compositeImageRaw.select('Human_Disturbance').lt(0)
toggledCropland = compositeImageRaw.select('cropland').lt(0)
toggledGrazing = compositeImageRaw.select('grazing').lt(0)
toggledPasture = compositeImageRaw.select('pasture').lt(0)
toggledRangeland = compositeImageRaw.select('rangeland').lt(0)
toggledWDPA = compositeImageRaw.select('WDPA').gte(0)
# load the potential tree cover and rename it to 'PresentTreeCover'
potentialTreeCover = ee.Image('users/leonidmoore/Potential_Forest_Cover_Adjusted').rename("PresentTreeCover")

In [12]:
# define the list of retained predictors
retainedPropeties = ['Aridity_Index',
                      'CHELSA_Annual_Mean_Temperature',
                      'CHELSA_Annual_Precipitation',
                      'CHELSA_Isothermality',
                      'CHELSA_Max_Temperature_of_Warmest_Month',
                      'CHELSA_Mean_Diurnal_Range',
                      'CHELSA_Mean_Temperature_of_Coldest_Quarter',
                      'CHELSA_Mean_Temperature_of_Driest_Quarter',
                      'CHELSA_Mean_Temperature_of_Warmest_Quarter',
                      'CHELSA_Mean_Temperature_of_Wettest_Quarter',
                      'CHELSA_Min_Temperature_of_Coldest_Month',
                      'CHELSA_Precipitation_Seasonality',
                      'CHELSA_Precipitation_of_Coldest_Quarter',
                      'CHELSA_Precipitation_of_Driest_Month',
                      'CHELSA_Precipitation_of_Driest_Quarter',
                      'CHELSA_Precipitation_of_Warmest_Quarter',
                      'CHELSA_Precipitation_of_Wettest_Month',
                      'CHELSA_Precipitation_of_Wettest_Quarter',
                      'CHELSA_Temperature_Annual_Range',
                      'CHELSA_Temperature_Seasonality',
                      'Depth_to_Water_Table',
                      'EarthEnvTopoMed_Eastness',
                      'EarthEnvTopoMed_Elevation',
                      'EarthEnvTopoMed_Northness',
                      'EarthEnvTopoMed_ProfileCurvature',
                      'EarthEnvTopoMed_Roughness',
                      'EarthEnvTopoMed_Slope',
                      'SG_Absolute_depth_to_bedrock',
                      'WorldClim2_SolarRadiation_AnnualMean',
                      'WorldClim2_WindSpeed_AnnualMean',
                      'EarthEnvCloudCover_MODCF_interannualSD',
                      'EarthEnvCloudCover_MODCF_intraannualSD',
                      'EarthEnvCloudCover_MODCF_meanannual',
                      'EarthEnvTopoMed_AspectCosine',
                      'EarthEnvTopoMed_AspectSine',
                      'SG_Clay_Content_0_100cm',
                      'SG_Coarse_fragments_0_100cm',
                      'SG_Sand_Content_0_100cm',
                      'SG_Silt_Content_0_100cm',
                      'SG_Soil_pH_H2O_0_100cm']
print(retainedPropeties[0:5])

['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month']


In [13]:
# replace the human activity layers in the compositeImageRaw
compositeImage = compositeImageRaw.select(retainedPropeties).addBands(toggledCultivated).addBands(toggledUrban).addBands(toggledDisturbance).addBands(toggledCropland).addBands(toggledGrazing).addBands(toggledPasture).addBands(toggledRangeland).addBands(toggledWDPA).addBands(potentialTreeCover)
# present the composite band names
print(colored('The band names are:', 'blue', attrs=['bold']),compositeImage.bandNames().getInfo())

The band names are: ['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvTopoMed_Eastness', 'EarthEnvTopoMed_Elevation', 'EarthEnvTopoMed_Northness', 'EarthEnvTopoMed_ProfileCurvature', 'EarthEnvTopoMed_Roughness', 'EarthEnvTopoMed_Slope', 'SG_Absolute_depth

#### 2.2 Machine learning mapping

In [15]:
# define a loop through the seed list
seedList = np.arange(0, 100, 1).tolist()
print(colored('The models are:', 'blue', attrs=['bold']),seedList)
print(colored('Model is running:\nWith paramter sets:', 'blue', attrs=['bold']))
# for seed in seedList: range(0,len(seedList))
for seed in seedList:
    # load the points data with the covariates
    trainTable = ee.FeatureCollection('users/leonidmoore/ForestBiomass/RemoteSensingModel/TrainTables/Remote_Sensing_Random_Subsampled_Train_Table_seed_'+str(seed))
    # print(trainTable.size().getInfo())
    parameterTable = pd.read_csv('RemoteSensingModel/GridSearchResult/Remote_Sensing_Biomass_Modeling_Grid_Search_Seed_'+str(seed)+'.csv', float_precision='round_trip')
    # not recomend to run the code below
    # print(parameterTable.head())
    # extract the paramters
    variablesPerSplitVal = int(parameterTable['variablesPerSplit'].iat[0]) # mtry
    minLeafPopulationVal = int(parameterTable['minLeafPopulation'].iat[0]) # minrow
    maxNodesVal = int(parameterTable['maxNodes'].iat[0]) # mac depth
    print('seed',seed,variablesPerSplitVal,minLeafPopulationVal,maxNodesVal)
    # define the random forest classifier
    rfClassifier = ee.Classifier.smileRandomForest(numberOfTrees = 200,
                                                   variablesPerSplit = variablesPerSplitVal, # mtry
                                                   minLeafPopulation = minLeafPopulationVal, # minrow
                                                   maxNodes = maxNodesVal, # max depth
                                                   bagFraction = 0.632,
                                                   seed = seed).setOutputMode('REGRESSION')
    trainedClassifier = rfClassifier.train(features = trainTable,
                                           classProperty = 'RemoteBiomass',
                                           inputProperties = propertyOfInterest)
    # execute the prediction to generate the map
    potentialCarbonDensityMap = compositeImage.classify(trainedClassifier)
    # print(predictedWoodDensityMap.getInfo())
    predictionExport = ee.batch.Export.image.toAsset(image = potentialCarbonDensityMap,
                                                     description = '20210411_Potential_Cabon_Density_Map_To_Asset_'+str(seed),
                                                     assetId = 'users/leonidmoore/ForestBiomass/RemoteSensingModel/PredictedMaps/Predicted_Potential_Remote_Sensing_Biomass_Map_with_Seed_'+str(seed),
                                                     region = unboundedGeo,
                                                     crs = 'EPSG:4326',
                                                     crsTransform = [0.008333333333333333,0,-180,0,-0.008333333333333333,90],
                                                     maxPixels = 1e13)

    # print(predictionExportAsset)
    # start the export task
    predictionExport.start()
    # show the task status
    predictionExport.status()

The models are: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]
Model is running:
With paramter sets:
seed 0 18 8 90
seed 1 15 8 90
seed 2 18 14 80
seed 3 15 14 90
seed 4 18 10 70
seed 5 15 14 80
seed 6 18 12 90
seed 7 15 14 90
seed 8 18 18 90
seed 9 18 6 90
seed 10 15 12 90
seed 11 18 12 90
seed 12 18 12 90
seed 13 18 12 80
seed 14 15 14 90
seed 15 18 8 80
seed 16 15 8 90
seed 17 18 6 80
seed 18 15 10 90
seed 19 12 8 90
seed 20 18 6 90
seed 21 18 12 90
seed 22 18 14 90
seed 23 18 8 90
seed 24 18 14 80
seed 25 15 10 90
seed 26 15 10 80
seed 27 12 12 80
seed 28 15 14 90
seed 29 18 10 90
seed 30 15 6 90
seed 31 18 10 90
seed 32 18 6 90
seed 33 18 8 90